# 💼 The Analyst's Notebook · Part 6
### Every column the desk can offer

Part 5 ended with a model of one column, chosen by cross-validation, beating the persistence rule on 11 of the eleven instruments. It also ended with a question left open: the desk has far more information than Apple's own last twenty days. Six windows of its history, the direction of its recent returns, and the volatility of every other instrument on the desk are all known on the day the forecast is made.

Part 6 puts all of that in. Ordinary least squares makes the forecast worse, and then a penalty on the coefficients, chosen with the folds from Part 5, repairs it. Whether the repaired model beats the one-column model is a question with a different answer on Apple and on the desk as a whole, and the report at the end has to say both.

## How to work through this

- Run the **quick load** cell first. It brings back what Part 5 established and loads the price table.
- Each question builds on the last, so keep them in order and keep your variables. Later questions use the names earlier ones created.
- Cells with `...` are blanks. The notebook runs cleanly even before you fill them in, so **Run all** is always safe.
- Hints and solutions are folded under each question. Work first, then check.

**A note on units.** The lecture worked in percent. This notebook keeps the plain decimals of Parts 1 to 5, so an RMSE of `0.004` means 0.4 percentage points of daily volatility. One consequence matters today: the lasso's alpha scales with the target, so its grid here is a hundred times smaller than the lecture's. Ridge's does not, once the columns are standardised.

*Stuck for more than 15 minutes? Ask a friend, ask an AI for a hint (not the answer), or email me at `jobo@econ.au.dk`.*

---

## ⚙️ Quick load

The packages, the price table, and what Part 5 left you. Run it and read what it prints.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score, TimeSeriesSplit, GridSearchCV

CANDIDATE_DIRS = ["data", os.path.join("..", "data"), "."]
REPO_RAW_URL = "https://raw.githubusercontent.com/theill95/mlfin-2026/main/data/"   # used when the CSV files are not next to the notebook


def data_path(filename):
    """Where the course CSV files are, wherever you happen to be running."""
    for folder in CANDIDATE_DIRS:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    if REPO_RAW_URL is not None:
        return REPO_RAW_URL + filename
    raise FileNotFoundError(
        f"Could not find {filename}. Run this notebook from the course folder, "
        f"upload the CSV into Colab, or set REPO_RAW_URL."
    )


def rmse(actual, predicted):
    """Root mean squared error, as in the lecture, as a plain number."""
    return float(np.sqrt(mean_squared_error(actual, predicted)))


# The whole universe: eleven instruments, 2015 to 2024, returns in plain decimals
prices = pd.read_csv(data_path("prices.csv"), parse_dates=["date"])
wide = prices.pivot(index="date", columns="ticker", values="close")
rets = wide.pct_change()
TICKERS = sorted(prices["ticker"].unique())

folds = TimeSeriesSplit(n_splits=5)

# --- What Part 5 established ---
part5_target = "sd of daily returns over the next 20 trading days"
part5_features = ["vol_20d", "ret_20d", "up_20d"]     # the three candidates' columns
part5_chosen = ["vol_20d"]                            # the one cross-validation kept
part5_split = "by date: train to 2022-12-31, test from 2023-01-01"
part5_rmse = 0.00402          # the chosen model, on the test block
part5_pers_rmse = 0.00417     # repeat the last 20 days
part5_beat = 11               # instruments where the model beat persistence, of 11

print("Loaded prices:", prices.shape[0], "rows")
print("Instruments  :", ", ".join(TICKERS))
print()
print("Part 5 left you a chosen model:")
print("  target  :", part5_target)
print("  columns :", part5_chosen, "chosen from", part5_features)
print("  split   :", part5_split)
print(f"  test RMSE {part5_rmse:.5f}  against {part5_pers_rmse:.5f} for persistence")
print(f"  beats persistence on {part5_beat} of {len(TICKERS)} instruments")
print()
print("Today the desk hands you every column it has.")

---

### Q1 · Where Part 5 stopped

Rebuild Part 5's table for Apple, the three features and the target with incomplete rows dropped, split it at the end of 2022, and refit the chosen one-column model. Print its test RMSE and check it matches `part5_rmse`.

$$\text{vol\_next}_t = \text{sd}\big(r_{t+1},\, \ldots,\, r_{t+20}\big)$$

In [ ]:
table5 = ...
train5 = ...
test5 = ...

model5 = LinearRegression()
...

check = ...
print(check)
print('matches Part 5:', ...)

<details>
<summary>💡 Hint 1</summary>

The three features are `rets['AAPL'].rolling(20).std()`, `.rolling(20).mean()` and `(rets['AAPL'] > 0).rolling(20).mean()`; the target is the first of those with `.shift(-20)`.

</details>

<details>
<summary>💡 Hint 2</summary>

Fit on `train5[['vol_20d']]`, score on `test5`. `abs(check - part5_rmse) < 0.00001` is True when the two agree.

</details>

<details>
<summary>✅ Solution</summary>

```python
table5 = pd.DataFrame({
    'vol_20d': rets['AAPL'].rolling(20).std(),
    'ret_20d': rets['AAPL'].rolling(20).mean(),
    'up_20d': (rets['AAPL'] > 0).rolling(20).mean(),
})
table5['vol_next'] = rets['AAPL'].rolling(20).std().shift(-20)
table5 = table5.dropna()
train5 = table5.loc[:'2022-12-31']
test5 = table5.loc['2023-01-01':]

model5 = LinearRegression()
model5.fit(train5[['vol_20d']], train5['vol_next'])

check = rmse(test5['vol_next'], model5.predict(test5[['vol_20d']]))
print(check)
print('matches Part 5:', abs(check - part5_rmse) < 0.00001)
```

0.00402, and `True`. Everything in this part is measured against that number, so it is worth one cell to see it come back.

</details>

---

### Q2 · Every column the desk can offer

Build the wide table, `table`, from three loops: Apple's volatility over `[5, 10, 20, 40, 60, 120]` days as `vol_<w>d`, its average return over `[5, 20, 60]` days as `ret_<w>d`, then Part 4's `up_20d`, then the 20-day volatility of every **other** instrument as `<ticker>_vol`. Add the target and drop incomplete rows. Print the shape.

In [ ]:
table = pd.DataFrame()

for w in [5, 10, 20, 40, 60, 120]:
    ...

for w in [5, 20, 60]:
    ...

table['up_20d'] = ...

for t in TICKERS:
    ...

table['vol_next'] = ...
table = ...
print(...)

<details>
<summary>💡 Hint 1</summary>

Column names are text built from the number: `'vol_' + str(w) + 'd'`. Inside the last loop, `if t != 'AAPL':`.

</details>

<details>
<summary>💡 Hint 2</summary>

The target is `rets['AAPL'].rolling(20).std().shift(-20)`, then `.dropna()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
table = pd.DataFrame()

for w in [5, 10, 20, 40, 60, 120]:
    table['vol_' + str(w) + 'd'] = rets['AAPL'].rolling(w).std()

for w in [5, 20, 60]:
    table['ret_' + str(w) + 'd'] = rets['AAPL'].rolling(w).mean()

table['up_20d'] = (rets['AAPL'] > 0).rolling(20).mean()

for t in TICKERS:
    if t != 'AAPL':
        table[t + '_vol'] = rets[t].rolling(20).std()

table['vol_next'] = rets['AAPL'].rolling(20).std().shift(-20)
table = table.dropna()
print(table.shape)
```

2,376 rows and 21 columns: twenty features and the target. The row count fell from Part 5's 2,476 because the 120-day window needs a hundred more days to fill. That matters for the next question.

</details>

---

### Q3 · The same split, and a fair one-column baseline

Split `table` at the end of 2022 into `train` and `test`, and store the twenty feature names in `columns`. Then refit the one-column model on the **new** training rows and store its test RMSE as `one_rmse`. Why refit rather than reuse `part5_rmse`?

In [ ]:
train = ...
test = ...
columns = ...

one_model = LinearRegression()
...
one_rmse = ...
print(one_rmse)

<details>
<summary>💡 Hint 1</summary>

`list(table.columns[:-1])` is every column but the target.

</details>

<details>
<summary>💡 Hint 2</summary>

The rows changed in Q2, so the fair comparison is a one-column model fitted and scored on exactly the rows the wide models will use.

</details>

<details>
<summary>✅ Solution</summary>

```python
train = table.loc[:'2022-12-31']
test = table.loc['2023-01-01':]
columns = list(table.columns[:-1])

one_model = LinearRegression()
one_model.fit(train[['vol_20d']], train['vol_next'])
one_rmse = rmse(test['vol_next'], one_model.predict(test[['vol_20d']]))
print(one_rmse)
```

0.00412, a little above Part 5's 0.00402 because the training block lost its first hundred days. Every comparison from here on is against this number, on these rows.

</details>

---

### Q4 · Everything in

Fit ordinary least squares on all twenty columns as `ols_model`. Print its training RMSE and its test RMSE next to the one-column model's, and store the test RMSE as `ols_rmse`. Then print the error of guessing the training average.

In [ ]:
ols_model = LinearRegression()
...

print('one column, train:', ...)
print('all columns, train:', ...)
print('one column, test :', ...)
ols_rmse = ...
print('all columns, test :', ...)

print('guess the average:', ...)

<details>
<summary>💡 Hint 1</summary>

Training RMSE scores predictions on `train` against `train['vol_next']`.

</details>

<details>
<summary>💡 Hint 2</summary>

The average guess is `np.full(len(test), train['vol_next'].mean())`.

</details>

<details>
<summary>✅ Solution</summary>

```python
ols_model = LinearRegression()
ols_model.fit(train[columns], train['vol_next'])

print('one column, train:', rmse(train['vol_next'], one_model.predict(train[['vol_20d']])))
print('all columns, train:', rmse(train['vol_next'], ols_model.predict(train[columns])))
print('one column, test :', one_rmse)
ols_rmse = rmse(test['vol_next'], ols_model.predict(test[columns]))
print('all columns, test :', ols_rmse)

print('guess the average:', rmse(test['vol_next'], np.full(len(test), train['vol_next'].mean())))
```

Training error falls from 0.00728 to 0.00642; test error rises from 0.00412 to 0.00563. **The twenty-column OLS is worse than guessing the average**, which scores 0.00533. Nineteen extra columns did not add information the model could use; they added freedom to fit noise, and it used all of it.

</details>

---

### Q5 · Count the wrong signs

Every **volatility** column in this table should, if anything, raise the forecast when it rises. Count the negative coefficients in `ols_model` with a mask, then print their names with a loop. Which of them are volatility columns?

In [ ]:
n_negative = ...
print(n_negative)

...

<details>
<summary>💡 Hint</summary>

`(ols_model.coef_ < 0).sum()` counts. Then `for name, b in zip(columns, ols_model.coef_):` with `if b < 0: print(name)` inside.

</details>

<details>
<summary>✅ Solution</summary>

```python
n_negative = (ols_model.coef_ < 0).sum()
print(n_negative, 'of', len(columns))

for name, b in zip(columns, ols_model.coef_):
    if b < 0:
        print(name)
```

10 of the twenty are negative. The two return columns are allowed to be, since a falling market is a volatile one. The other 8 are volatility columns, `vol_20d` and `vol_60d` among them: more than half of the desk's volatility columns lower the forecast of Apple's volatility, which no one believes. The columns are near-copies of each other, and OLS is trading coefficient between them.

</details>

---

### Q6 · The lecture's alpha, on raw columns

Fit `Ridge(alpha=1000)` on the twenty raw columns and print its test RMSE. It will look familiar. Which number from Q4 does it match, and why?

In [ ]:
raw_ridge = ...
...
print(...)

<details>
<summary>💡 Hint 1</summary>

Fit and predict exactly as with `LinearRegression`.

</details>

<details>
<summary>💡 Hint 2</summary>

The columns are in decimals, so their values are around 0.01. A coefficient of 1 on such a column moves the forecast by 0.01, and alpha 1000 charges 1000 for it. Every slope is crushed and the forecast is the intercept.

</details>

<details>
<summary>✅ Solution</summary>

```python
raw_ridge = Ridge(alpha=1000)
raw_ridge.fit(train[columns], train['vol_next'])
print(rmse(test['vol_next'], raw_ridge.predict(test[columns])))
```

0.00532, which is the average guess from Q4 to four decimals. The largest coefficient left is 0.0004. In decimals every column is a hundred times smaller than in the lecture, so it needs a coefficient a hundred times larger, which the penalty charges ten thousand times more for. This is the units problem, and it is why the next step is not a smaller alpha.

</details>

---

### Q7 · Standardise, then penalise

Build a `Pipeline` with a `StandardScaler` step named `'scale'` and a `Ridge(alpha=1000)` step named `'ridge'`. Fit it and print the test RMSE.

In [ ]:
pipe = Pipeline([...])
...
print(...)

<details>
<summary>💡 Hint</summary>

`Pipeline([('scale', StandardScaler()), ('ridge', Ridge(alpha=1000))])`, then `.fit` and `.predict` on the raw columns; the scaling happens inside.

</details>

<details>
<summary>✅ Solution</summary>

```python
pipe = Pipeline([('scale', StandardScaler()), ('ridge', Ridge(alpha=1000))])
pipe.fit(train[columns], train['vol_next'])
print(rmse(test['vol_next'], pipe.predict(test[columns])))
```

0.00460. The same alpha that flattened everything in Q6 now gives a forecast better than the twenty-column OLS (0.00563) and still worse than the single column (0.00412). With every column in standard deviations, the penalty judges columns by their usefulness rather than their units.

</details>

---

### Q8 · Choose alpha on the folds

Loop over `[1, 10, 100, 1000, 10000]`: build the pipeline with that alpha, cross-validate it on the training rows with `folds`, and store the mean RMSE in `cv_by_alpha`. Print the dictionary and the best alpha.

In [ ]:
cv_by_alpha = {}

for alpha in [1, 10, 100, 1000, 10000]:
    ...

print(cv_by_alpha)
print('best:', ...)

<details>
<summary>💡 Hint 1</summary>

`cross_val_score(pipe, train[columns], train['vol_next'], cv=folds, scoring='neg_root_mean_squared_error')`, then `-scores.mean()`.

</details>

<details>
<summary>💡 Hint 2</summary>

`min(cv_by_alpha, key=cv_by_alpha.get)` is the key with the smallest value.

</details>

<details>
<summary>✅ Solution</summary>

```python
cv_by_alpha = {}

for alpha in [1, 10, 100, 1000, 10000]:
    pipe = Pipeline([('scale', StandardScaler()), ('ridge', Ridge(alpha=alpha))])
    scores = cross_val_score(pipe, train[columns], train['vol_next'],
                             cv=folds, scoring='neg_root_mean_squared_error')
    cv_by_alpha[alpha] = round(float(-scores.mean()), 5)

print(cv_by_alpha)
print('best:', min(cv_by_alpha, key=cv_by_alpha.get))
```

Best at 1000, the same alpha the lecture found in percent. Scaling the target by a hundred scales both the residual sum of squares and the squared coefficients by ten thousand, so the trade-off between them is unchanged. That is a property of ridge; it is not true of the lasso.

</details>

---

### Q9 · GridSearchCV, and the test block once

Let `GridSearchCV` do Q8: the pipeline with `Ridge()` and no alpha, the grid `{'ridge__alpha': [1, 10, 100, 1000, 10000]}`, `folds`, and the RMSE scoring. Fit it as `search`, print the best alpha and score, then predict the test rows **once** and store the RMSE as `ridge_rmse`.

In [ ]:
pipe = Pipeline([('scale', StandardScaler()), ('ridge', Ridge())])
grid = ...

search = ...
...

print(...)
print(...)

ridge_rmse = ...
print('test:', ridge_rmse, ' one column:', one_rmse)

<details>
<summary>💡 Hint 1</summary>

`GridSearchCV(pipe, grid, cv=folds, scoring='neg_root_mean_squared_error')`, then `.fit(train[columns], train['vol_next'])`.

</details>

<details>
<summary>💡 Hint 2</summary>

`search.predict(test[columns])` uses the best pipeline refitted on all the training rows.

</details>

<details>
<summary>✅ Solution</summary>

```python
pipe = Pipeline([('scale', StandardScaler()), ('ridge', Ridge())])
grid = {'ridge__alpha': [1, 10, 100, 1000, 10000]}

search = GridSearchCV(pipe, grid, cv=folds, scoring='neg_root_mean_squared_error')
search.fit(train[columns], train['vol_next'])

print(search.best_params_)
print(-search.best_score_)

ridge_rmse = rmse(test['vol_next'], search.predict(test[columns]))
print('test:', ridge_rmse, ' one column:', one_rmse)
```

Alpha 1000 at a cross-validated 0.00761, and 0.00460 on the test block against 0.00412 for the single column. On Apple, twenty penalised columns lose to one unpenalised column. The folds already said so: the one-column model cross-validates at 0.00755, below every row of the grid. The penalty repaired the damage of Q4; it did not turn the extra columns into signal.

</details>

---

### Q10 · Lasso, on its own scale

Search a lasso pipeline (step named `'lasso'`) over `{'lasso__alpha': [0.00001, 0.0001, 0.001, 0.01]}` as `lasso_search`. Print the best alpha and score, store the test RMSE as `lasso_rmse`, and print the names of the columns the winning lasso kept as the list `kept`.

In [ ]:
lasso_pipe = Pipeline([('scale', StandardScaler()), ('lasso', Lasso(max_iter=20000))])
lasso_grid = ...

lasso_search = ...
...

print(...)
print(...)
lasso_rmse = ...
print('test:', lasso_rmse)

coefs = ...
kept = []
...
print(kept)

<details>
<summary>💡 Hint 1</summary>

The lecture's grid ran from 0.001 to 1 in percent. The lasso's penalty is on $|\beta|$, which scales with the target, so in decimals the grid is a hundred times smaller.

</details>

<details>
<summary>💡 Hint 2</summary>

`lasso_search.best_estimator_.named_steps['lasso'].coef_` are the coefficients. Loop over `zip(columns, coefs)` and append `name` when `b != 0`.

</details>

<details>
<summary>✅ Solution</summary>

```python
lasso_pipe = Pipeline([('scale', StandardScaler()), ('lasso', Lasso(max_iter=20000))])
lasso_grid = {'lasso__alpha': [0.00001, 0.0001, 0.001, 0.01]}

lasso_search = GridSearchCV(lasso_pipe, lasso_grid, cv=folds, scoring='neg_root_mean_squared_error')
lasso_search.fit(train[columns], train['vol_next'])

print(lasso_search.best_params_)
print(-lasso_search.best_score_)
lasso_rmse = rmse(test['vol_next'], lasso_search.predict(test[columns]))
print('test:', lasso_rmse)

coefs = lasso_search.best_estimator_.named_steps['lasso'].coef_
kept = []
for name, b in zip(columns, coefs):
    if b != 0:
        kept.append(name)
print(kept)
```

Alpha 0.001 at 0.00776, and 0.00448 on the test block: a little better than ridge, still behind the single column. It keeps 8 of the twenty columns: vol_5d, vol_10d, vol_20d, vol_120d, ret_5d, ret_20d, DIS_vol, MSFT_vol. `max_iter=20000` is there because small alphas need more passes than the default, and the convergence warning would otherwise appear.

</details>

---

### Q11 · The kept set, fold by fold

Fit the winning lasso (its alpha is in `lasso_search.best_params_`) on the fitting rows of each fold and print the columns it keeps each time. Is the list from Q10 stable?

In [ ]:
best_alpha = ...

...

<details>
<summary>💡 Hint 1</summary>

`best_alpha = lasso_search.best_params_['lasso__alpha']`. Then `for fit_rows, score_rows in folds.split(train):`, with `block = train.iloc[fit_rows]` and a fresh pipeline fitted on the block.

</details>

<details>
<summary>💡 Hint 2</summary>

Collect the kept names with the same loop as Q10 and print the list.

</details>

<details>
<summary>✅ Solution</summary>

```python
best_alpha = lasso_search.best_params_['lasso__alpha']

for fit_rows, score_rows in folds.split(train):
    block = train.iloc[fit_rows]
    fold_lasso = Pipeline([('scale', StandardScaler()), ('lasso', Lasso(alpha=best_alpha, max_iter=20000))])
    fold_lasso.fit(block[columns], block['vol_next'])
    fold_kept = []
    for name, b in zip(columns, fold_lasso.named_steps['lasso'].coef_):
        if b != 0:
            fold_kept.append(name)
    print(fold_kept)
```

Five different lists, with only `DIS_vol` in every one. The lasso's zeros say which near-copy it happened to keep on those rows, not which columns carry information. Report the kept list as a compact forecast, never as a finding about the columns.

</details>

---

### Q12 · Both penalties

Search an elastic net pipeline (step `'enet'`) over `alpha` in `[0.0001, 0.001, 0.01]` and `l1_ratio` in `[0.1, 0.5, 0.9]` as `enet_search`. Print the best pair and score, and store the test RMSE as `enet_rmse`.

In [ ]:
enet_pipe = Pipeline([('scale', StandardScaler()), ('enet', ElasticNet(max_iter=20000))])
enet_grid = {...}

enet_search = ...
...

print(...)
print(...)
enet_rmse = ...
print('test:', enet_rmse)

<details>
<summary>💡 Hint</summary>

Two keys in one dictionary: `{'enet__alpha': [...], 'enet__l1_ratio': [...]}`. Nine combinations, forty-five fits.

</details>

<details>
<summary>✅ Solution</summary>

```python
enet_pipe = Pipeline([('scale', StandardScaler()), ('enet', ElasticNet(max_iter=20000))])
enet_grid = {'enet__alpha': [0.0001, 0.001, 0.01], 'enet__l1_ratio': [0.1, 0.5, 0.9]}

enet_search = GridSearchCV(enet_pipe, enet_grid, cv=folds, scoring='neg_root_mean_squared_error')
enet_search.fit(train[columns], train['vol_next'])

print(enet_search.best_params_)
print(-enet_search.best_score_)
enet_rmse = rmse(test['vol_next'], enet_search.predict(test[columns]))
print('test:', enet_rmse)
```

Alpha 0.001 with l1_ratio 0.9, at 0.00775 on the folds and 0.00447 on the test block, keeping 8 columns. Three penalised models within a hair of each other, all behind the single column on Apple. The folds ranked them the same way, which is the point of having the folds.

</details>

---

### Q13 · Across the desk

Apple is one instrument. Write `penalised_vs_one(ticker)`: build the wide table for that ticker (its own windows, `up_20d`, the other ten instruments' volatility), split at the end of 2022, fit the one-column model, run the ridge grid search from Q9, and return the pair `(ridge_rmse, one_rmse)` on the test block. Run it for every ticker into a dictionary `desk` and count the wins.

In [ ]:
def penalised_vs_one(ticker):
    ...

desk = {}
for ticker in TICKERS:
    ...

wins = ...
print('penalised wins on', ..., 'of', ...)

<details>
<summary>💡 Hint 1</summary>

The function is Q2, Q3 and Q9 with `ticker` in place of `'AAPL'`, including in the `if` that skips the instrument itself.

</details>

<details>
<summary>💡 Hint 2</summary>

`sum(1 for t in desk if desk[t][0] < desk[t][1])` counts the wins. Eleven grid searches take a little while.

</details>

<details>
<summary>✅ Solution</summary>

```python
def penalised_vs_one(ticker):
    frame = pd.DataFrame()
    for w in [5, 10, 20, 40, 60, 120]:
        frame['vol_' + str(w) + 'd'] = rets[ticker].rolling(w).std()
    for w in [5, 20, 60]:
        frame['ret_' + str(w) + 'd'] = rets[ticker].rolling(w).mean()
    frame['up_20d'] = (rets[ticker] > 0).rolling(20).mean()
    for t in TICKERS:
        if t != ticker:
            frame[t + '_vol'] = rets[t].rolling(20).std()
    frame['vol_next'] = rets[ticker].rolling(20).std().shift(-20)
    frame = frame.dropna()

    tr = frame.loc[:'2022-12-31']
    te = frame.loc['2023-01-01':]
    cols = list(frame.columns[:-1])

    one = LinearRegression()
    one.fit(tr[['vol_20d']], tr['vol_next'])
    pipe = Pipeline([('scale', StandardScaler()), ('ridge', Ridge())])
    grid_search = GridSearchCV(pipe, {'ridge__alpha': [1, 10, 100, 1000, 10000]},
                               cv=folds, scoring='neg_root_mean_squared_error')
    grid_search.fit(tr[cols], tr['vol_next'])

    return (rmse(te['vol_next'], grid_search.predict(te[cols])),
            rmse(te['vol_next'], one.predict(te[['vol_20d']])))

desk = {}
for ticker in TICKERS:
    desk[ticker] = penalised_vs_one(ticker)

wins = sum(1 for t in desk if desk[t][0] < desk[t][1])
print('penalised wins on', wins, 'of', len(desk))
```

7 of 11. The penalised wide model beats the single column on most of the desk, by 13% on DIS, and loses on AAPL, KO, PG, XOM. Apple, the instrument every part of this case has been built on, is one of the exceptions. A result checked on one stock is one observation.

</details>

---

### Q14 · Calm days and busy days

Split Apple's test block with a mask, days where `vol_20d` is below its median and the rest, and print the RMSE of both the ridge `search` and `one_model` on each half. Does the wide model help where it matters?

In [ ]:
ridge_errors = ...
one_errors = ...
calm = ...

print('calm, ridge:', ...)
print('calm, one  :', ...)
print('busy, ridge:', ...)
print('busy, one  :', ...)

<details>
<summary>💡 Hint 1</summary>

`test['vol_next'] - search.predict(test[columns])` and the same with `one_model.predict(test[['vol_20d']])`.

</details>

<details>
<summary>💡 Hint 2</summary>

`calm = test['vol_20d'] < test['vol_20d'].median()`; `errors[~calm]` is the busy half; `np.sqrt((errors[calm] ** 2).mean())`.

</details>

<details>
<summary>✅ Solution</summary>

```python
ridge_errors = test['vol_next'] - search.predict(test[columns])
one_errors = test['vol_next'] - one_model.predict(test[['vol_20d']])
calm = test['vol_20d'] < test['vol_20d'].median()

print('calm, ridge:', round(np.sqrt((ridge_errors[calm] ** 2).mean()), 5))
print('calm, one  :', round(np.sqrt((one_errors[calm] ** 2).mean()), 5))
print('busy, ridge:', round(np.sqrt((ridge_errors[~calm] ** 2).mean()), 5))
print('busy, one  :', round(np.sqrt((one_errors[~calm] ** 2).mean()), 5))
```

Calm: 0.00433 against 0.00384. Busy: 0.00486 against 0.00438. The single column wins both halves on Apple, so there is no regime in which the wide model earns its place here. That is a cleaner statement than "it loses on average", and it took one mask.

</details>

---

### Q15 · Draw it

One figure: what happened in the test years as a line, the one-column forecast and the ridge forecast as two more lines, with a legend and a title.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
...
plt.show()

<details>
<summary>💡 Hint 1</summary>

`ax.plot(test.index, test['vol_next'], label='what happened')`, then the two forecasts with their own labels.

</details>

<details>
<summary>💡 Hint 2</summary>

`ax.legend()`, `ax.set_ylabel('20-day volatility')`, `ax.set_title(..., loc='left')`.

</details>

<details>
<summary>✅ Solution</summary>

```python
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(test.index, test['vol_next'], label='what happened', color='black', linewidth=1.2)
ax.plot(test.index, one_model.predict(test[['vol_20d']]), label='one column')
ax.plot(test.index, search.predict(test[columns]), label='twenty columns, ridge')
ax.set_ylabel('20-day volatility')
ax.legend()
ax.set_title('Apple, 2023 to 2024: two forecasts against what happened', loc='left')
plt.show()
```

The two forecasts track each other closely and both lag the turns, as every model of this kind does. The wide model is not doing anything different from the narrow one on Apple; it is doing the same thing with more noise.

</details>

---

### Q16 · Write down what you would defend

Finish the way Parts 4 and 5 finished: one dictionary and a function that prints it with a verdict. Fill in `report`, then write `summarise(report)`: it prints each entry on its own line and ends with one sentence on whether the wide model replaces the one-column model **on Apple**, and one on the desk as a whole.

In [ ]:
report = {
    'target': ...,
    'columns_offered': ...,
    'alpha': ...,
    'chosen_by': ...,
    'one_column_rmse': ...,
    'ols_rmse': ...,
    'ridge_rmse': ...,
    'lasso_rmse': ...,
    'desk_wins': ...,
}

def summarise(report):
    ...

summarise(report)

<details>
<summary>💡 Hint 1</summary>

Most values are already in variables: `len(columns)`, `search.best_params_['ridge__alpha']`, `one_rmse`, `ols_rmse`, `ridge_rmse`, `lasso_rmse`, and `wins` from Q13.

</details>

<details>
<summary>💡 Hint 2</summary>

Inside the function, `for key in report:` then `print(f'{key:18} {report[key]}')`. The verdicts are two `if`s: one on `ridge_rmse < one_column_rmse`, one on `desk_wins`.

</details>

<details>
<summary>✅ Solution</summary>

```python
report = {
    'target': part5_target,
    'columns_offered': len(columns),
    'alpha': search.best_params_['ridge__alpha'],
    'chosen_by': 'grid search on five time-ordered folds, training rows only',
    'one_column_rmse': round(one_rmse, 5),
    'ols_rmse': round(ols_rmse, 5),
    'ridge_rmse': round(ridge_rmse, 5),
    'lasso_rmse': round(lasso_rmse, 5),
    'desk_wins': f'{wins} of {len(desk)}',
}

def summarise(report):
    """Print a finished comparison, and say what replaces what."""
    for key in report:
        print(f'{key:18} {report[key]}')

    if report['ridge_rmse'] < report['one_column_rmse']:
        print('\nOn Apple, the penalised wide model replaces the one-column model.')
    else:
        print('\nOn Apple, the one-column model stays. The extra columns did not help.')
    print(f"Across the desk, the penalised wide model wins on {report['desk_wins']} instruments.")

summarise(report)
```

Nine lines and two verdicts, and they point in different directions. That is the honest state of things: on Apple the twenty columns were tested and did not help, and on most of the desk they did. Note that `ols_rmse` is in the report too. A reader deserves to know what the columns cost before the penalty, because it says how much of the wide model's score is the penalty's doing.

</details>

---

## 🧭 What you have now

| what | where it lives |
|:--|:--|
| Part 5's model, rebuilt and checked | `table5`, `model5` |
| the wide table and its split | `table`, `train`, `test`, `columns` |
| the fair one-column baseline | `one_model`, `one_rmse` |
| what OLS does with twenty columns | `ols_model`, `ols_rmse` |
| the alphas on the folds | `cv_by_alpha` |
| the ridge search, refitted | `search`, `ridge_rmse` |
| the lasso search and its kept columns | `lasso_search`, `lasso_rmse`, `kept` |
| the elastic net search | `enet_search`, `enet_rmse` |
| the whole desk | `desk`, `wins` |
| the calm and busy halves | `calm` |
| the thing you would defend | `report` |

## What changed since Part 5

- **The model got every column the desk has.** Twenty instead of one. Unpenalised, that was worse than guessing the average (0.00563 against 0.00533). With a penalty chosen on the folds it was 0.00460, which repairs the damage without beating the single column on Apple.
- **The units problem showed up in earnest.** The lecture's alpha on raw decimal columns flattened the model to the average (Q6). Standardising inside a pipeline brought the lecture's alpha back as the right one (Q8), because ridge's trade-off does not depend on the target's units. The lasso's does, which is why its grid moved by a factor of a hundred (Q10).
- **The zeros are not findings.** Q11 showed the lasso keeping a different set of columns on every fold block.
- **The desk disagrees with Apple.** Q13 found the penalised wide model winning on 7 of 11 instruments. The report carries both numbers.

## Where this leaves the risk report

Six parts ago this was two stocks compared with a subtraction. It is now a forecast with a chosen model, a documented search over a setting, a baseline that beats it on the stock it was built on, and a desk-wide result that says the extra columns are worth having on most of the others.

The tools from this part carry forward unchanged: a pipeline, a grid, the folds, and the test block opened once. Every later model in the course has settings of its own, and they are chosen exactly this way.

**Next block:** the target becomes a label rather than a number, and the same penalty appears in a classifier under a different name.